[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/finique/praxis-reproductions/blob/main/358063-futures-dynamic-momentum.ipynb)

# Trend-Following Strategies via Dynamic Momentum Learning (2021) — Praxis reproduction

**Bruno P. C. Levy & Hedibert F. Lopes** · [arXiv 2106.08420](https://arxiv.org/abs/2106.08420) · 56 futures contracts, 1980–2020

**The claim:** replacing the naive time-series-momentum sign rule with a dynamic-logistic binary classifier — 127 models, one per subset of momentum look-backs (1–12 months), sequentially reweighted by predictive record (DMA/DMS) — lifts a diversified futures portfolio from **Sharpe 0.81 to 1.23** net of costs.

**What this notebook does:** rebuilds the strategy from public data with the same vendor our validation engine uses (Yahoo front-month continuous futures, 39 of the 56 contracts — including the paper's full 25-commodity sleeve) and reproduces our report's exact numbers: **DMS-TVP 0.49 vs naive-12m 0.21** on 2003-09→2020-09, the +274bp/yr lift (t = 0.85, never significant), the paper's 2009 momentum-crash escape (which replicates almost to the number), and the post-publication window where the lift reverses (0.42 vs 0.60).

Same data and logic → our exact numbers; the full C0–C7 gate suite (deflation, PBO, crisis, frictions, spanning) lives in the report: [market-graphs.com/praxis/papers/358063](https://market-graphs.com/praxis/papers/358063).

*Runtime: ~10 minutes on a free Colab CPU (the 127-model filter runs for 39 contracts twice — time-varying and constant-parameter). The sample is pinned at 2026-06-30; the filter is fully deterministic (no random numbers).*

In [ ]:
# Pinned to the engine's versions so the filter arithmetic reproduces bit-for-bit.
%pip install -q numpy==2.1.2 pandas==2.2.3 yfinance==0.2.65

In [ ]:
"""Praxis reproduction — paper 358063 (arXiv 2106.08420), dynamic momentum.

Self-contained port of the engine blocks (data/futures_monthly.py,
blocks/dma_logistic.py, blocks/tsmom.py, exemplars/paper_358063.py).
Deterministic — no RNG anywhere; identical input data → identical output.

Timing convention (leak-free, verified in the engine by a look-ahead trap that
inflates a peeking naive-1m rule to Sharpe 7.8): row t of the predictor matrix
holds momentum through month t−1; the classifier predicts s_t = 1{r_t ≥ 0};
positions for month t are formed at the end of t−1; realised-s_t updates happen
strictly after the prediction is recorded.
"""
from itertools import combinations

import numpy as np
import pandas as pd

PAPER_END, T6_START, FRONTIER_END = "2020-09-30", "2010-07-31", "2026-06-30"
LOOKBACKS = (1, 2, 4, 6, 8, 10, 12)
LAMBDA_TVP, LAMBDA_CP, ALPHA_GRID = (0.98, 0.99, 1.0), (1.0,), (0.99, 1.0)
PRIOR_VAR, _CLIP = 100.0, 1e-8
SIGMA_TARGET, EWMA_DELTA, PPY, MIN_ASSETS, TRAINING_MONTHS = 0.40, 0.97, 12, 10, 36
COST_BPS = {"equity": 1.0, "bond": 1.0, "fx": 1.0, "commodity": 3.0}

UNIVERSE = [  # (yahoo, name, class, rolls/yr)
    ("CL=F","Light crude oil","commodity",12),("BZ=F","Brent crude oil","commodity",12),
    ("HO=F","Heating oil","commodity",12),("NG=F","Natural gas","commodity",12),
    ("RB=F","RBOB gasoline","commodity",12),("HG=F","Copper","commodity",5),
    ("GC=F","Gold","commodity",5),("PA=F","Palladium","commodity",4),
    ("PL=F","Platinum","commodity",4),("SI=F","Silver","commodity",5),
    ("GF=F","Feeder cattle","commodity",8),("LE=F","Live cattle","commodity",6),
    ("HE=F","Lean hogs","commodity",7),("ZC=F","Corn","commodity",5),
    ("ZO=F","Oats","commodity",5),("ZL=F","Soybean oil","commodity",7),
    ("ZM=F","Soybean meal","commodity",7),("ZS=F","Soybeans","commodity",7),
    ("ZW=F","Wheat","commodity",5),("CC=F","Cocoa","commodity",5),
    ("KC=F","Coffee","commodity",5),("CT=F","Cotton #2","commodity",5),
    ("LBS=F","Lumber","commodity",6),("OJ=F","Orange juice","commodity",6),
    ("SB=F","Sugar #11","commodity",4),
    ("ES=F","S&P500","equity",4),("NQ=F","Nasdaq 100","equity",4),("YM=F","Dow","equity",4),
    ("ZT=F","UST 2y","bond",4),("ZF=F","UST 5y","bond",4),
    ("ZN=F","UST 10y","bond",4),("ZB=F","UST 30y","bond",4),
    ("6E=F","EUR/USD","fx",4),("6J=F","JPY/USD","fx",4),("6B=F","GBP/USD","fx",4),
    ("6A=F","AUD/USD","fx",4),("6C=F","CAD/USD","fx",4),("6S=F","CHF/USD","fx",4),
    ("6N=F","NZD/USD","fx",4),
]


def load_panel(end=FRONTIER_END):
    import yfinance as yf
    closes, klass, rolls = {}, {}, {}
    for yahoo, name, cls, rpy in UNIVERSE:
        df = yf.download(yahoo, period="max", interval="1d",
                         progress=False, auto_adjust=False)
        if df is None or df.empty:
            continue
        close = df["Close"]
        if isinstance(close, pd.DataFrame):
            close = close.iloc[:, 0]
        s = close.astype(float)
        s.index = pd.to_datetime(s.index).tz_localize(None)
        closes[name], klass[name], rolls[name] = s[s > 0], cls, rpy
    daily = pd.DataFrame(closes).sort_index().loc[:end]
    monthly = daily.resample("ME").last()
    for name in monthly.columns:                      # dead contracts stay dead
        last = daily[name].dropna().index.max()
        monthly.loc[monthly.index > last, name] = np.nan
    rets, logrets = monthly.pct_change(fill_method=None), np.log(monthly).diff()
    eligible = pd.DataFrame(False, index=monthly.index, columns=monthly.columns)
    for n in monthly.columns:
        gate = monthly[n].dropna().index.min() + pd.DateOffset(months=TRAINING_MONTHS)
        eligible.loc[(eligible.index >= gate) & monthly[n].notna(), n] = True
    return {"returns": rets, "log_returns": logrets, "klass": klass,
            "rolls_per_year": rolls, "eligible": eligible}


# ----------------------------- dynamic-logistic DMA/DMS (McCormick et al.) ----
def model_space(K):
    return [c for r in range(1, K + 1) for c in combinations(range(K), r)]


def momentum_matrix(lr, lookbacks=LOOKBACKS):
    T, K = len(lr), len(lookbacks)
    X = np.full((T, K), np.nan)
    csum = np.concatenate([[0.0], np.nancumsum(lr)])
    ok = np.isfinite(lr)
    for i, L in enumerate(lookbacks):
        for t in range(L, T):
            if ok[t - L:t].all():
                X[t, i] = csum[t] - csum[t - L]       # momentum through t−1
    return X


def _sigmoid(z):
    return 1.0 / (1.0 + np.exp(-np.clip(z, -35.0, 35.0)))


def _lse(v):
    m = np.max(v)
    return m + np.log(np.sum(np.exp(v - m)))


def _sym(A):
    return 0.5 * (A + np.swapaxes(A, -1, -2))


def run_dma(X, s, mode="tvp", prior_var=PRIOR_VAR):
    lambdas = LAMBDA_TVP if mode == "tvp" else LAMBDA_CP
    T, K = X.shape
    models = model_space(K)
    M, D = len(models), K + 1
    mask = np.zeros((M, D)); mask[:, 0] = 1.0
    for j, sub in enumerate(models):
        for k in sub:
            mask[j, 1 + k] = 1.0
    omask = np.einsum("mi,mj->mij", mask, mask)
    m = np.zeros((M, D))
    C = np.tile(np.eye(D) * prior_var, (M, 1, 1))
    log_pi = np.full(M, -np.log(M))
    alpha_cur = ALPHA_GRID[-1]
    eyeD = np.eye(D)
    out_dma, out_dms = np.full(T, np.nan), np.full(T, np.nan)

    for t in range(T):
        if not np.isfinite(X[t]).all() or not np.isfinite(s[t]):
            continue
        x = np.concatenate([[1.0], X[t]]) * mask
        p_pred = np.clip(_sigmoid(np.einsum("md,md->m", x, m)), _CLIP, 1 - _CLIP)
        lp = log_pi * alpha_cur; lp -= _lse(lp)
        pi_pred = np.exp(lp)
        out_dma[t] = float(pi_pred @ p_pred)
        out_dms[t] = float(p_pred[int(np.argmax(pi_pred))])

        st = float(s[t])
        best_ld = np.full(M, -np.inf)
        best_m, best_C = m.copy(), C.copy()
        for lam in lambdas:
            Rm = (C / lam) * omask + (prior_var * eyeD) * (1 - omask)
            w = p_pred * (1 - p_pred)
            Rinv = np.linalg.inv(_sym(Rm) + 1e-12 * eyeD)
            H = -(w[:, None, None] * np.einsum("mi,mj->mij", x, x)) - Rinv
            Cn = np.linalg.inv(_sym(-H) + 1e-12 * eyeD)
            mn = m + np.einsum("mij,mj->mi", Cn, x * (st - p_pred)[:, None])
            p_n = np.clip(_sigmoid(np.einsum("md,md->m", x, mn)), _CLIP, 1 - _CLIP)
            ll = st * np.log(p_n) + (1 - st) * np.log(1 - p_n)
            diff = mn - m
            quad = np.einsum("mi,mij,mj->m", diff, Rinv, diff)
            ld = (0.5 * np.linalg.slogdet(_sym(Cn))[1] + ll
                  - 0.5 * np.linalg.slogdet(_sym(Rm))[1] - 0.5 * quad)
            better = ld > best_ld
            best_ld = np.where(better, ld, best_ld)
            best_m = np.where(better[:, None], mn, best_m)
            best_C = np.where(better[:, None, None], Cn, best_C)
        m, C = best_m, best_C

        dens = np.exp(best_ld - np.max(best_ld))      # α selection, paper Eq 16
        best_score = -np.inf
        for al in ALPHA_GRID:
            lpa = log_pi * al; lpa -= _lse(lpa)
            sc = float(np.exp(lpa) @ dens)
            if sc > best_score:
                best_score, alpha_cur = sc, al
        lp = log_pi * alpha_cur; lp -= _lse(lp)
        lp = lp + best_ld
        log_pi = lp - _lse(lp)
    return out_dma, out_dms


# ------------------------------------------------- portfolio (paper Eq 18/19) --
def ewma_vol(returns, delta=EWMA_DELTA, ppy=PPY, min_obs=12):
    out = pd.DataFrame(np.nan, index=returns.index, columns=returns.columns)
    for c in returns.columns:
        mean = var = None; n = 0
        for t, x in returns[c].items():
            if np.isfinite(x):
                if mean is None:
                    mean, var = x, 0.0
                else:
                    dev = x - mean
                    mean = delta * mean + (1 - delta) * x
                    var = delta * var + (1 - delta) * dev * dev
                n += 1
                if n >= min_obs and var > 0:
                    out.loc[t, c] = np.sqrt(var * ppy)
    return out


def sleeve_mask(panel, sleeve):
    elig = panel["eligible"]
    if sleeve != "all":
        elig = elig.copy()
        elig.loc[:, [c for c in elig.columns if panel["klass"][c] != sleeve]] = False
    out = elig.copy()
    out.loc[elig.sum(axis=1) < MIN_ASSETS, :] = False
    return out


def run_portfolio(panel, signals, sleeve, vol):
    elig = sleeve_mask(panel, sleeve)
    v = vol.shift(1)
    lever = (SIGMA_TARGET / v).where(np.isfinite(v))
    live = signals.notna() & lever.notna() & panel["returns"].notna() & elig
    n_t = live.sum(axis=1)
    w = (signals * lever).where(live).div(n_t.replace(0, np.nan), axis=0)
    gross = (w * panel["returns"]).sum(axis=1, min_count=1)
    wf = w.fillna(0.0)
    bps = pd.Series({c: COST_BPS[panel["klass"][c]] for c in w.columns})
    rolls = pd.Series({c: panel["rolls_per_year"][c] for c in w.columns})
    cost = (((wf - wf.shift(1)).abs() * bps).sum(axis=1)
            + (wf.abs() * bps * (rolls / 12.0) * 2.0).sum(axis=1)) / 1e4
    return (gross - cost.reindex(gross.index).fillna(0.0)).dropna()


def scale10(r):
    sd = r.std(ddof=1) * np.sqrt(PPY)
    return r * (0.10 / sd) if sd > 0 else r


def sharpe(r):
    r = r.dropna()
    return float(r.mean() / r.std(ddof=1) * np.sqrt(PPY))


In [ ]:
print("Fetching 39 futures chains from Yahoo (~1 min)…")
panel = load_panel()
lr = panel["log_returns"]
vol = ewma_vol(panel["returns"])
print(f"panel: {lr.shape[1]} contracts · {lr.index.min():%Y-%m} → {lr.index.max():%Y-%m}")

print("\nRunning the 127-model dynamic classifier per contract (TVP + CP, ~8 min)…")
sig = {}
for mode in ("tvp", "cp"):
    dma = pd.DataFrame(np.nan, index=lr.index, columns=lr.columns)
    dms = dma.copy()
    for name in lr.columns:
        r = lr[name].values
        X = momentum_matrix(r)
        s = np.where(np.isfinite(r), (r >= 0).astype(float), np.nan)
        a, b = run_dma(X, s, mode=mode)
        dma[name], dms[name] = a, b
    sig[(mode, "dma")] = dma.where(dma.isna(), np.where(dma >= 0.5, 1.0, -1.0))
    sig[(mode, "dms")] = dms.where(dms.isna(), np.where(dms >= 0.5, 1.0, -1.0))
    print(f"  {mode} done")

# naive 12m benchmark: sign of momentum through t−1
naive = pd.DataFrame(np.nan, index=lr.index, columns=lr.columns)
for name in lr.columns:
    m12 = momentum_matrix(lr[name].values, (12,))[:, 0]
    naive[name] = np.where(np.isfinite(m12), np.where(m12 >= 0, 1.0, -1.0), np.nan)

print("\nBuilding portfolios (net of per-class costs, Eq 18/19 sizing)…")
port = {k: run_portfolio(panel, v, "all", vol) for k, v in sig.items()}
port[("naive", 12)] = run_portfolio(panel, naive, "all", vol)
cmd = {k: run_portfolio(panel, v, "commodity", vol)
       for k, v in [(("tvp", "dms"), sig[("tvp", "dms")]), (("naive", 12), naive)]}

rows = []
for key, label, claim_full, claim_t6 in [
        (("tvp", "dms"), "DMS-TVP (headline)", 1.23, 1.10),
        (("cp", "dms"), "DMS-CP", 1.17, 1.18),
        (("tvp", "dma"), "DMA-TVP", 1.06, 0.94),
        (("naive", 12), "Naive 12m TSMOM", 0.81, 0.50)]:
    r = port[key]
    rows.append({
        "strategy": label, "claimed (1980-2020)": claim_full,
        "measured ≤2020-09": round(sharpe(scale10(r.loc[:PAPER_END])), 2),
        "claimed T6": claim_t6,
        "measured T6": round(sharpe(scale10(r.loc[T6_START:PAPER_END])), 2),
        "post-pub 2020-10→2026-06": round(sharpe(scale10(r.loc["2020-10-01":])), 2)})
print("\n=== claimed → measured (net Sharpe, ex-post 10% vol presentation) ===")
print(pd.DataFrame(rows).to_string(index=False))

f, n = port[("tvp", "dms")], port[("naive", 12)]
d = (scale10(f.loc[:PAPER_END]) - scale10(n.loc[:PAPER_END])).dropna()
print(f"\nThe claimed lift, measured: {d.mean()*12*1e4:+.0f} bp/yr, "
      f"t = {d.mean()/d.std(ddof=1)*np.sqrt(len(d)):.2f} (n={len(d)} months) — never significant")

for key, label in [(("tvp", "dms"), "DMS-TVP"), (("naive", 12), "naive-12m")]:
    w = scale10(port[key].loc[:PAPER_END]).loc["2009-03-01":"2010-06-30"]
    print(f"2009 momentum crash (2009m03–2010m06) {label}: {((1+w).prod()-1)*100:+.1f}% "
          f"cumulative (paper: DMS-TVP +22.1 · naive −25.3)")

print(f"\ncommodity sleeve (25/25 contracts) ≤2020-09: "
      f"DMS-TVP {sharpe(scale10(cmd[('tvp','dms')].loc[:PAPER_END])):.2f} vs "
      f"naive {sharpe(scale10(cmd[('naive',12)].loc[:PAPER_END])):.2f} "
      f"(paper Fig 1: ≈1.08 vs ≈0.44)")


In [ ]:
import matplotlib.pyplot as plt

eq_f = (1 + scale10(port[("tvp", "dms")])).cumprod()
eq_n = (1 + scale10(port[("naive", 12)])).cumprod()
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(eq_f.index, eq_f.values, label="DMS-TVP dynamic momentum (measured 0.49)", lw=1.3)
ax.plot(eq_n.index, eq_n.values, label="Naive 12-month TSMOM (measured 0.21)", lw=1.3)
ax.axvline(pd.Timestamp(PAPER_END), color="gray", ls=":", lw=1)
ax.text(pd.Timestamp(PAPER_END), ax.get_ylim()[0], "  paper sample ends /\n  post-publication →",
        fontsize=8, color="gray", va="bottom")
ax.axvspan(pd.Timestamp("2009-03-01"), pd.Timestamp("2010-06-30"), alpha=0.12, color="crimson",
           label="2009 momentum crash (the dodge that replicates)")
ax.set_yscale("log"); ax.set_ylabel("Growth of 1 (both at 10% vol, net, log scale)")
ax.set_title("Paper 358063 — the claimed Sharpe 1.23 vs what public data shows (0.49 vs naive 0.21)")
ax.legend(loc="upper left", fontsize=8); ax.grid(alpha=0.25)
plt.tight_layout(); plt.show()
